# 3. 系统增强

这一节关注"系统级"问题：当单次检索流程已经优化过，仍需处理多轮记忆、多文档路由、工具编排与可恢复执行时，如何构建更稳健的 RAG 系统。

## 本章要解决什么

前两章解决的是单次请求内的问题——上下文不全、流程不够。但当系统需要记住上一轮对话、协调多个数据源、或在复杂任务中自主规划执行路径时，单次请求的优化已经不够了。

本章关注的是**跨请求的工程问题**：状态怎么延续、工具怎么组织、复杂任务怎么规划与执行。我们会从最基础的记忆管理开始，逐步引入多文档路由、知识图谱，最后用 Agentic RAG 把这些能力组装成一个完整的自主系统。

## 流程增强 vs 系统增强

流程增强关注"单次请求内"的多步决策（多轮检索、子问题拆解、质量把关）。

系统增强关注"跨请求"的工程问题：
- 上一轮对话的信息怎么延续？-> Memory
- 多个数据源怎么组织和路由？-> Multi-Document Agent
- 复杂任务怎么规划、执行、反思？-> Agentic RAG

判断标准：如果去掉 history / 去掉多文档路由 / 去掉任务规划，系统仍能回答，那是流程问题；否则是系统问题。


## 统一实验设置

与前两节保持一致：

- **数据**：`../3. 索引阶段/data/pumpkin_book.pdf`（南瓜书《机器学习公式详解》）
- **问答数据**：`../3. 索引阶段/data/train_dataset.json`（选取前 5 个问答对用于实验）
- **生成模型**：`glm-4-flash-250414`
- **向量模型**：本地 `BAAI/bge-small-zh-v1.5`
- **评估**：使用 LLM 作为裁判进行评估

## 环境准备

本节使用智谱 AI 的 `GLM-4-Flash` 做生成模型，使用本地 `BAAI/bge-small-zh-v1.5` 做 embedding。运行前请确保：

1. 安装依赖：`pip install langchain langchain-community langchain-chroma zhipuai python-dotenv pymupdf pandas modelscope sentence-transformers transformers torch`
2. 在项目根目录的 `.env` 文件中配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载本地 embedding 模型到当前目录下的 `./models/`


In [ ]:
import os
import re
import json
import warnings
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import sys
sys.path.insert(0, ".")
from _common import (
    get_embeddings, get_cleaned_pdf_documents, open_or_build_chroma,
    llm_call as _raw_llm_call, build_rag_generation_prompt,
    trim_context_to_budget, simple_eval_2pt, build_compare_table,
    CHROMA_COLLECTION, CONTEXT_CHAR_BUDGET, PDF_PATH, QA_PATH,
)

warnings.filterwarnings("ignore")


def llm_call(prompt: str) -> str:
    """6.3 节默认每次成功调用后 sleep 1 秒。"""
    return _raw_llm_call(prompt, sleep_after=1.0)


def load_chunks(chunk_size=256, chunk_overlap=20):
    docs = list(get_cleaned_pdf_documents())
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""], keep_separator=True,
    )
    return splitter.split_documents(docs)


def build_retriever(chunk_size=256, chunk_overlap=20, k=4, persist_subdir="baseline_256_20"):
    persist_dir = f"./chroma_db/{persist_subdir}"
    chunks = load_chunks(chunk_size, chunk_overlap)
    ids = [f"b{i}" for i in range(len(chunks))]
    vs = open_or_build_chroma(persist_dir, chunks, ids)
    return vs.as_retriever(search_kwargs={"k": k})


retriever = build_retriever()
print("✅ 6.3 环境准备完成（公共底座来自 _common）")


## 本节钩子：`class XxxSystem: ask(q) -> str`

6.1 钩子是 `*_context`（变 context 拼法），6.2 钩子是 `*_pipeline`（变流程编排）。
但系统增强的核心是 **状态**——跨轮记忆、跨源路由——单题接口 `q -> str` 根本
表达不出来。所以本节钩子升一层：每个方法是一个 **带状态的类**，对外提供 `ask(q)`。

```python
class MemoryRAGSystem:    def ask(q) -> str   # 内部更新 history
class MultiDocAgent:      def ask(q) -> str   # 内部路由到不同 source
```

公共底座为此提供 `run_session_eval(system, qna_dict)`：按顺序调用 `system.ask`，
**复用同一 system 实例** 以保留状态。它是 6.3 首次出现的概念，下面会先 inline
完整定义一次，再放进 `_common.py` 供未来扩展使用。


## Memory：多轮对话 RAG

多轮对话最大的麻烦是 **指代**——"它的候选划分点是怎么确定的？" 这里的"它"指什么？
单看一句话，retriever 完全无法定位。`MemoryRAGSystem` 的做法是：每次 `ask(q)` 时，
先用 LLM 把 `(history, q)` 改写成一个 **不依赖历史** 的独立查询（condensed query），
再用这个独立查询去 retrieve + generate，最后把 `(原始 q, 答案)` 入 history。

这种"先改写、再检索"的模式在工业实现中被称为 **Condense Question Chain**。

In [ ]:
def run_session_eval(system, qna_dict, *, eval_prompt_template=None):
    """按顺序调 system.ask(q)，复用同一 system 实例以保留状态。"""
    rows = []
    for question, expected in qna_dict.items():
        answer = system.ask(question)
        score = simple_eval_2pt(
            answer, expected, question, prompt_template=eval_prompt_template,
        )
        rows.append({
            "question": question, "llm_answer": answer,
            "expected_answer": expected, "rag_eval_results": score,
        })
    return pd.DataFrame(rows)


MEMORY_BEHAVIORAL_EVAL_PROMPT = (
    "你是判卷人。本题是多轮对话中的一轮，「用户问题」可能含指代（如『它』、"
    "『上面提到的方法』）。请按 0~2 分评估「模型答案」：\n"
    "2 分：正确解析指代，并给出与「参考答案」核心一致的回答。\n"
    "1 分：解析了指代但答案部分缺失或方向正确但不完整。\n"
    "0 分：未解析指代、答非所问、或与参考答案明显矛盾。\n\n"
    "用户问题：{question}\n参考答案：{expected_answer}\n模型答案：{llm_answer}\n\n"
    "仅输出一行，只包含字符 0、1 或 2。"
)


> 上面 `run_session_eval` 已收纳到 `_common.py`，本节后续与未来章节通过 `import` 复用。


In [ ]:
class MemoryRAGSystem:
    """多轮对话 RAG：每次 ask 时，先用 LLM 把 (history, q) 改写成独立 query，
    再正常 retrieve + generate；新 (q, a) 入 history。"""

    def __init__(self, retriever, max_history: int = 5):
        self.retriever = retriever
        self.history: list[tuple[str, str]] = []
        self.max_history = max_history
        self.last_debug: dict = {}

    def _condense(self, q: str) -> str:
        if not self.history:
            return q
        hist_text = "\n".join(f"Q: {hq}\nA: {ha}" for hq, ha in self.history)
        prompt = (
            "下面是历史对话。请把最新的问题改写成一个不依赖历史的独立问题，"
            "保留所有指代消解后的实体名。只输出改写后的问题，不要任何前缀。\n\n"
            f"历史：\n{hist_text}\n\n最新问题：{q}\n\n改写："
        )
        return llm_call(prompt).strip()

    def ask(self, q: str) -> str:
        condensed = self._condense(q)
        docs = self.retriever.invoke(condensed)
        ctx = trim_context_to_budget(
            "\n\n".join(d.page_content for d in docs),
            CONTEXT_CHAR_BUDGET,
        )
        ans = llm_call(build_rag_generation_prompt(q, ctx))
        self.last_debug = {
            "original": q, "condensed": condensed,
            "n_hits": len(docs),
            "first_hit_head": docs[0].page_content[:80] if docs else "",
        }
        self.history.append((q, ans))
        self.history = self.history[-self.max_history :]
        return ans


### 单 session 观察：condensed 改写发生了吗？

先只跑第 1 个 session（决策树连续属性），打印每轮的 `condensed` 与命中变化。
我们关心的客观信号是：从 turn 2 开始，`condensed` 应该不等于原问题
（因为原问题"它的..."独立看是无意义的，必须被改写）。

In [ ]:
def inspect_memory_session(session: dict) -> None:
    print(f"━━━━━ Session: {session['title']} ━━━━━\n")
    sys = MemoryRAGSystem(retriever)
    for i, turn in enumerate(session["turns"]):
        q = turn["q"]
        ans = sys.ask(q)
        d = sys.last_debug
        print(f"--- Turn {i+1} ---")
        print(f"  原问题：{q}")
        print(f"  Condensed：{d['condensed']}")
        print(f"  改写发生：{'是' if d['condensed'] != q else '否'}")
        print(f"  命中数：{d['n_hits']}；首条命中：{d['first_hit_head']}...")
        print(f"  答案：{ans[:160]}...")
        print()


with open("data/memory_sessions.json", "r", encoding="utf-8") as f:
    SESSIONS = json.load(f)["sessions"]

inspect_memory_session(SESSIONS[0])


### 行为评估：memory vs no-memory baseline

在 inspect 之外，还需要一个量化对比。对照组 `NoMemorySystem` 不维护 history，
每题独立检索；它在 turn 2/3 上 **应当** 因为读不懂指代而失分。我们用
`MEMORY_BEHAVIORAL_EVAL_PROMPT`（针对指代解析的 0~2 分判卷）跑全部 3 个 session，
合并后用 `build_compare_table` 出对比表。

In [ ]:
def session_to_qna(session):
    return {turn["q"]: turn["a"] for turn in session["turns"]}


def baseline_no_memory_ask_factory():
    """对照组：每题独立 retrieve + generate，无 history。"""
    class NoMemorySystem:
        def __init__(self, retriever):
            self.retriever = retriever
        def ask(self, q):
            docs = self.retriever.invoke(q)
            ctx = trim_context_to_budget(
                "\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET,
            )
            return llm_call(build_rag_generation_prompt(q, ctx))
    return NoMemorySystem(retriever)


memory_dfs, baseline_dfs = [], []
for sess in SESSIONS:
    qna = session_to_qna(sess)
    mem_sys = MemoryRAGSystem(retriever)
    base_sys = baseline_no_memory_ask_factory()
    memory_dfs.append(run_session_eval(mem_sys, qna, eval_prompt_template=MEMORY_BEHAVIORAL_EVAL_PROMPT))
    baseline_dfs.append(run_session_eval(base_sys, qna, eval_prompt_template=MEMORY_BEHAVIORAL_EVAL_PROMPT))

memory_all = pd.concat(memory_dfs, ignore_index=True)
baseline_all = pd.concat(baseline_dfs, ignore_index=True)
memory_compare = build_compare_table(
    [baseline_all, memory_all],
    names=["no_memory_baseline", "memory"],
)
memory_compare


### Memory 表怎么读

- `no_memory_baseline` 列：在 turn 2/3 上低分意味着模型把"它"、"上面提到"等指代
  当成实体来检索，召回到无关章节或答非所问。
- `memory` 列：高分需要两个条件同时满足——condensed 改写正确 + 改写后的查询能命中。
- 如果两列都偏低，往往是 retriever k 不够或 chunk 切得太碎，需要回到 6.1/6.2 调；
  Memory 解决的是 **指代消解**，不是检索召回的根本问题。